# LSTM Đa Biến với Entity Embedding

**Target**: `Units Sold`  
**Approach**: Entity Embedding cho 4 static categorical features (`Store ID`, `Product ID`, `Category`, `Region`) + numerical features (bao gồm `Weather Condition`, `Seasonality` dạng số) → LSTM multi-step forecast  
**Horizons**: 7, 14, 28 ngày

In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')
import os
import kagglehub


tf.random.set_seed(42)
np.random.seed(42)

DATA_DIR = kagglehub.dataset_download("atomicd/retail-store-inventory-and-demand-forecasting")
DATA_PATH = os.path.join(DATA_DIR, "sales_data.csv")

#DATA_PATH  = "/kaggle/input/datasets/thaonngyn/retail-store-inventory-and-demand-forecasting/sales_data.csv"
RESULT_DIR = "/kaggle/working/"
TARGET     = 'Units Sold'
TRAIN_END  = '2023-06-30'
VAL_END    = '2023-10-31'
HORIZONS   = [7, 14, 28]
LOOKBACK   = 30
LAG        = 7

2026-03-20 04:48:42.572020: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773982122.878289      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773982122.967097      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773982123.704166      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773982123.704213      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773982123.704217      55 computation_placer.cc:177] computation placer alr

## 1. Load & Feature Engineering

In [2]:
df_raw = pd.read_csv(DATA_PATH)
df_raw['Date'] = pd.to_datetime(df_raw['Date'])
df_raw = df_raw.sort_values(['Store ID', 'Product ID', 'Date']).reset_index(drop=True)

# ── Static categorical → entity embedding ────────────────────────────────────
CAT_COLS = ['Store ID', 'Product ID', 'Category', 'Region']
cat_vocabs = {}
for col in CAT_COLS:
    uniq = sorted(df_raw[col].unique())
    cat_vocabs[col] = {v: i for i, v in enumerate(uniq)}
    df_raw[col + '_enc'] = df_raw[col].map(cat_vocabs[col])

# ── Time-varying categoricals → ordinal encode vào numerical sequence ────────
WEATHER_MAP = {'Sunny': 0, 'Cloudy': 1, 'Rainy': 2, 'Snowy': 3, 'Windy': 4, 'Stormy': 5}
SEASON_MAP  = {'Winter': 0, 'Spring': 1, 'Summer': 2, 'Fall': 3}
df_raw['weather_enc']  = df_raw['Weather Condition'].map(WEATHER_MAP).fillna(0).astype(int)
df_raw['season_enc']   = df_raw['Seasonality'].map(SEASON_MAP).fillna(0).astype(int)

# ── Numerical features ────────────────────────────────────────────────────────
grp = df_raw.groupby(['Store ID', 'Product ID'])[TARGET]
df_raw['lag_7']           = grp.shift(7)
df_raw['lag_14']          = grp.shift(14)
df_raw['lag_28']          = grp.shift(28)
df_raw['rolling_mean_7']  = grp.transform(lambda x: x.shift(1).rolling(7).mean())
df_raw['rolling_mean_14'] = grp.transform(lambda x: x.shift(1).rolling(14).mean())
df_raw['day_of_week']     = df_raw['Date'].dt.dayofweek
df_raw['day_of_month']    = df_raw['Date'].dt.day
df_raw['month']           = df_raw['Date'].dt.month
df_raw['is_weekend']      = (df_raw['day_of_week'] >= 5).astype(int)
df_raw = df_raw.bfill().fillna(0)

NUM_COLS = [
    TARGET,
    'Price', 'Discount', 'Competitor Pricing',
    'Inventory Level', 'Units Ordered',
    'Promotion', 'Epidemic',
    'weather_enc', 'season_enc',          # time-varying — giữ trong sequence
    'lag_7', 'lag_14', 'lag_28',
    'rolling_mean_7', 'rolling_mean_14',
    'day_of_week', 'day_of_month', 'month', 'is_weekend',
]
TARGET_IDX = NUM_COLS.index(TARGET)
ENC_COLS   = [c + '_enc' for c in CAT_COLS]

series_keys = sorted(df_raw.groupby(['Store ID', 'Product ID']).groups.keys())
print(f'Series: {len(series_keys)} | Num features: {len(NUM_COLS)} | Static cat: {len(CAT_COLS)}')
print('Vocab sizes:', {c: len(v) for c, v in cat_vocabs.items()})

Series: 100 | Num features: 19 | Static cat: 4
Vocab sizes: {'Store ID': 5, 'Product ID': 20, 'Category': 5, 'Region': 4}


## 2. Model với Entity Embedding

In [3]:
from tensorflow.keras import layers, models

def embedding_dim(vocab_size):
    """Rule of thumb: min(50, (vocab_size + 1) // 2)"""
    return min(50, (vocab_size + 1) // 2)


def build_model(
    lookback,
    n_num,
    cat_vocab_sizes,
    horizon,
    lstm_units=64,
    dropout=0.2,
    activation="tanh"   
):
    """
    Inputs:
      - num_input : (batch, lookback, n_num)
      - cat_input_i: (batch,)
    """

    num_input = layers.Input(shape=(lookback, n_num), name='num_input')

    cat_inputs, cat_embeds = [], []

    for i, (col, vocab_size) in enumerate(cat_vocab_sizes.items()):
        inp = layers.Input(shape=(1,), name=f'cat_{i}', dtype='int32')

        emb = layers.Embedding(
            vocab_size,
            embedding_dim(vocab_size),
            name='emb_' + col.replace(' ', '_')
        )(inp)

        emb = layers.Flatten()(emb)

        cat_inputs.append(inp)
        cat_embeds.append(emb)

    if cat_embeds:
        cat_concat = (
            layers.Concatenate()(cat_embeds)
            if len(cat_embeds) > 1 else cat_embeds[0]
        )

        cat_tiled = layers.RepeatVector(lookback)(cat_concat)
        x = layers.Concatenate(axis=-1)([num_input, cat_tiled])
    else:
        x = num_input

    x = layers.LSTM(lstm_units, return_sequences=True)(x)
    x = layers.Dropout(dropout)(x)

    x = layers.LSTM(lstm_units // 2)(x)
    x = layers.Dropout(dropout)(x)


    x = layers.Dense(64, activation=activation)(x)   
    x = layers.Dense(32, activation=activation)(x)

    out = layers.Dense(horizon)(x)

    model = models.Model(
        inputs=[num_input] + cat_inputs,
        outputs=out
    )

    return model

# Vocab sizes theo thứ tự CAT_COLS
cat_vocab_sizes = {col: len(cat_vocabs[col]) for col in CAT_COLS}

# Preview model
demo = build_model(LOOKBACK, len(NUM_COLS), cat_vocab_sizes, horizon=7)
demo.summary()

2026-03-20 04:49:16.002659: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ cat_0 (InputLayer)  │ (None, 1)         │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cat_1 (InputLayer)  │ (None, 1)         │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cat_2 (InputLayer)  │ (None, 1)         │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cat_3 (InputLayer)  │ (None, 1)         │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ emb_Store_ID        │ (None, 1, 3)      │         15 │ cat_0[0][0]       │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ emb_Product_ID      │ (None, 1, 10)     │        200 │ cat_1[0][0]       │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ emb_Category        │ (None, 1, 3)      │         15 │ cat_2[0][0]       │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ emb_Region          │ (None, 1, 2)      │          8 │ cat_3[0][0]       │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 3)         │          0 │ emb_Store_ID[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_1 (Flatten) │ (None, 10)        │          0 │ emb_Product_ID[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_2 (Flatten) │ (None, 3)         │          0 │ emb_Category[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_3 (Flatten) │ (None, 2)         │          0 │ emb_Region[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 18)        │          0 │ flatten[0][0],    │
│ (Concatenate)       │                   │            │ flatten_1[0][0],  │
│                     │                   │            │ flatten_2[0][0],  │
│                     │                   │            │ flatten_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ num_input           │ (None, 30, 19)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ repeat_vector       │ (None, 30, 18)    │          0 │ concatenate[0][0] │
│ (RepeatVector)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 30, 37)    │          0 │ num_input[0][0],  │
│ (Concatenate)       │                   │            │ repeat_vector[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ (None, 30, 64)    │     26,112 │ concatenate_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 30, 64)    │          0 │ lstm[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ (None, 32)        │     12,416 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 32)        │          0 │ lstm_1[0][0]    

 Total params: 43,189 (168.71 KB)

 Trainable params: 43,189 (168.71 KB)

 Non-trainable params: 0 (0.00 B)

In [4]:
import numpy as np
import tensorflow as tf
import random
from tensorflow.keras import layers, models, callbacks


SUBSET_SERIES = series_keys[:10]
H_TUNE = [7, 14, 28]

GA_SPACE = {
    "units1": [16, 32, 64],
    "dropout": [0.1, 0.2, 0.3],
    "lr": [0.0005, 0.001, 0.003],
    "batch": [16, 32],
    "lookback": [14, 30],
    "activation": ["relu", "tanh", "sigmoid"]
}


def embedding_dim(vocab_size):
    return min(50, (vocab_size + 1) // 2)


def build_model(lookback, n_num, cat_vocab_sizes, horizon,
                lstm_units=64, dropout=0.2, activation="relu"):

    num_input = layers.Input(shape=(lookback, n_num), name='num_input')

    cat_inputs, cat_embeds = [], []
    for i, (col, vocab_size) in enumerate(cat_vocab_sizes.items()):
        inp = layers.Input(shape=(1,), name=f'cat_{i}', dtype='int32')
        emb = layers.Embedding(vocab_size, embedding_dim(vocab_size))(inp)
        emb = layers.Flatten()(emb)
        cat_inputs.append(inp)
        cat_embeds.append(emb)

    if cat_embeds:
        cat_concat = layers.Concatenate()(cat_embeds) if len(cat_embeds) > 1 else cat_embeds[0]
        cat_tiled = layers.RepeatVector(lookback)(cat_concat)
        x = layers.Concatenate(axis=-1)([num_input, cat_tiled])
    else:
        x = num_input

    x = layers.LSTM(lstm_units)(x)
    x = layers.Dropout(dropout)(x)

    x = layers.Dense(64, activation=activation)(x)
    x = layers.Dropout(dropout)(x)

    out = layers.Dense(horizon)(x)

    return models.Model(inputs=[num_input] + cat_inputs, outputs=out)


def init_candidate():
    return {k: random.choice(v) for k, v in GA_SPACE.items()}


def mutate_candidate(sol, rate=0.2):
    new = sol.copy()
    for k in new:
        if random.random() < rate:
            new[k] = random.choice(GA_SPACE[k])
    return new


def crossover(parent1, parent2):
    return {k: random.choice([parent1[k], parent2[k]]) for k in parent1}


def tournament_selection(population, scores, k=2):  # nhỏ lại cho pop=4
    selected = random.sample(list(zip(population, scores)), k)
    selected = sorted(selected, key=lambda x: x[1])
    return selected[0][0]


def evaluate_candidate(params):

    tf.keras.backend.clear_session()
    horizon_scores = []

    for horizon in H_TUNE:

        X_num_tr, X_cats_tr, y_tr, X_num_vl, X_cats_vl, y_vl, scalers = \
            build_global_arrays(horizon, params["lookback"])

        model = build_model(
            params["lookback"],
            len(NUM_COLS),
            cat_vocab_sizes,
            horizon=horizon,
            lstm_units=params["units1"],
            dropout=params["dropout"],
            activation=params["activation"]
        )

        model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=params["lr"]),
            loss='mse'
        )

        cb = callbacks.EarlyStopping(
            monitor='val_loss',
            patience=3,
            restore_best_weights=True
        )

        model.fit(
            [X_num_tr] + X_cats_tr, y_tr,
            validation_data=([X_num_vl] + X_cats_vl, y_vl),
            epochs=20,
            batch_size=params["batch"],
            callbacks=[cb],
            verbose=0
        )

        smapes = []
        for store, product in SUBSET_SERIES:
            key = f'{store}_{product}'

            r = rolling_eval(
                model,
                scalers[key],
                store,
                product,
                horizon,
                params["lookback"]
            )

            smapes.append(r['smape'])

        horizon_scores.append(np.mean(smapes))

    return float(np.mean(horizon_scores))


def gao_search():

    n_generations = 4
    pop_size = 4
    elite_size = 1   

    population = [init_candidate() for _ in range(pop_size)]
    scores = [evaluate_candidate(p) for p in population]

    history = []

    for gen in range(n_generations):

        print(f"\n=== Generation {gen+1}/4 ===")

        idx = np.argsort(scores)
        population = [population[i] for i in idx]
        scores     = [scores[i] for i in idx]

        print(f"Best SMAPE: {scores[0]:.4f}")
        print(f"Best params: {population[0]}")

        history.append((population[0], scores[0]))

        new_population = population[:elite_size]

        while len(new_population) < pop_size:

            parent1 = tournament_selection(population, scores)
            parent2 = tournament_selection(population, scores)

            child = crossover(parent1, parent2)
            child = mutate_candidate(child)

            new_population.append(child)

        population = new_population
        scores = [evaluate_candidate(p) for p in population]

    best_idx = np.argmin(scores)

    return population[best_idx], scores[best_idx], history


## 3. Build Dataset

In [5]:
def make_sequences(num_arr, cat_row, target_idx, lookback, horizon, stride=7):
    """
    num_arr  : (T, n_num) scaled numerical
    cat_row  : (n_cat,)   encoded categorical values (static for this series)
    Returns  : X_num (N, lookback, n_num), X_cats list of (N,), y (N, horizon)
    """
    X_num, y = [], []
    for i in range(lookback, len(num_arr) - horizon + 1, stride):
        X_num.append(num_arr[i - lookback:i])
        y.append(num_arr[i:i + horizon, target_idx])
    X_num = np.array(X_num, dtype=np.float32)
    y     = np.array(y, dtype=np.float32)
    n     = len(X_num)
    X_cats = [np.full(n, cat_row[j], dtype=np.int32) for j in range(len(cat_row))]
    return X_num, X_cats, y


def build_global_arrays(horizon, lookback):
    X_num_tr, X_num_vl = [], []
    X_cats_tr = [[] for _ in CAT_COLS]
    X_cats_vl = [[] for _ in CAT_COLS]
    y_tr, y_vl = [], []
    scalers = {}

    for store, product in series_keys:
        sdf = df_raw[(df_raw['Store ID'] == store) & (df_raw['Product ID'] == product)]
        sdf = sdf.set_index('Date')
        key = f'{store}_{product}'

        num_data = sdf[NUM_COLS].values.astype(np.float32)
        cat_row  = sdf[ENC_COLS].iloc[0].values.astype(np.int32)

        train_num = sdf[:TRAIN_END][NUM_COLS].values.astype(np.float32)
        val_num   = sdf[:VAL_END][NUM_COLS].values.astype(np.float32)

        scaler = StandardScaler().fit(train_num)
        scalers[key] = scaler

        tr_scaled  = scaler.transform(train_num)
        val_scaled = scaler.transform(val_num)

        Xn_tr, Xc_tr, yt = make_sequences(tr_scaled,  cat_row, TARGET_IDX, lookback, horizon)
        Xn_vl, Xc_vl, yv = make_sequences(val_scaled, cat_row, TARGET_IDX, lookback, horizon)

        X_num_tr.append(Xn_tr); X_num_vl.append(Xn_vl)
        y_tr.append(yt);        y_vl.append(yv)
        for j in range(len(CAT_COLS)):
            X_cats_tr[j].append(Xc_tr[j])
            X_cats_vl[j].append(Xc_vl[j])

    X_num_tr = np.concatenate(X_num_tr)
    X_num_vl = np.concatenate(X_num_vl)
    y_tr     = np.concatenate(y_tr)
    y_vl     = np.concatenate(y_vl)
    X_cats_tr = [np.concatenate(x) for x in X_cats_tr]
    X_cats_vl = [np.concatenate(x) for x in X_cats_vl]

    return X_num_tr, X_cats_tr, y_tr, X_num_vl, X_cats_vl, y_vl, scalers


print('Dataset builder ready.')

Dataset builder ready.


## 4. Rolling Evaluation

In [6]:
def rolling_eval(model, scaler, store, product, horizon, lookback):
    sdf = df_raw[(df_raw['Store ID'] == store) & (df_raw['Product ID'] == product)].set_index('Date')
    cat_row = sdf[ENC_COLS].iloc[0].values.astype(np.int32)

    full_scaled = scaler.transform(sdf[NUM_COLS].values.astype(np.float32))
    eval_start  = pd.Timestamp(VAL_END) + pd.Timedelta(days=1)
    eval_end    = sdf.index.max()

    all_fc, all_ac = [], []
    t = eval_start
    while t + pd.Timedelta(days=horizon - 1) <= eval_end:
        t_loc     = sdf.index.get_loc(t)
        win_start = t_loc - lookback
        if win_start < 0:
            t += pd.Timedelta(days=horizon); continue
        actual = sdf[TARGET][t: t + pd.Timedelta(days=horizon - 1)].values
        if len(actual) < horizon:
            t += pd.Timedelta(days=horizon); continue

        X_num  = full_scaled[win_start:t_loc][np.newaxis]  # (1, lookback, n_num)
        X_cats = [np.array([[cat_row[j]]], dtype=np.int32).reshape(1,1) for j in range(len(CAT_COLS))]
        # model cat inputs expect shape (batch,) not (batch,1) — squeeze
        X_cats = [c.flatten() for c in X_cats]
        X_cats_in = [c[np.newaxis] for c in X_cats]  # (1,)

        fc_scaled = model.predict([X_num] + X_cats_in, verbose=0)[0]  # (horizon,)
        dummy = np.zeros((horizon, len(NUM_COLS)), dtype=np.float32)
        dummy[:, TARGET_IDX] = fc_scaled
        fc = np.clip(scaler.inverse_transform(dummy)[:, TARGET_IDX], 0, None)

        all_fc.append(fc)
        all_ac.append(actual.astype(np.float32))
        t += pd.Timedelta(days=horizon)

    if not all_fc:
        return {k: np.nan for k in ['smape', 'mase', 'rmse', 'rmsle']}

    fc_arr, ac_arr = np.array(all_fc), np.array(all_ac)
    train_vals = sdf[TARGET][:TRAIN_END].values.astype(np.float32)
    lag   = min(LAG, len(train_vals) - 1)
    denom = np.mean(np.abs(train_vals[lag:] - train_vals[:-lag])) or 1.0

    return {
        'smape': (2 * np.abs(fc_arr - ac_arr) / (np.abs(fc_arr) + np.abs(ac_arr) + 1e-8)).mean() * 100,
        'mase' : np.mean(np.abs(fc_arr - ac_arr)) / denom,
        'rmse' : float(np.sqrt(np.mean((fc_arr - ac_arr) ** 2))),
        'rmsle': float(np.sqrt(np.mean((np.log1p(np.clip(fc_arr, 0, None)) - np.log1p(np.clip(ac_arr, 0, None))) ** 2))),
    }


print('Evaluation function ready.')

Evaluation function ready.


## 5. Train & Evaluate

In [7]:
BEST, best_score, history = gao_search()

print("\nBest params:", BEST)
print("Best SMAPE:", best_score)


=== Generation 1/4 ===
Best SMAPE: 37.0818
Best params: {'units1': 16, 'dropout': 0.1, 'lr': 0.001, 'batch': 32, 'lookback': 30, 'activation': 'sigmoid'}

=== Generation 2/4 ===
Best SMAPE: 36.6297
Best params: {'units1': 16, 'dropout': 0.1, 'lr': 0.001, 'batch': 32, 'lookback': 30, 'activation': 'sigmoid'}

=== Generation 3/4 ===
Best SMAPE: 36.8662
Best params: {'units1': 64, 'dropout': 0.3, 'lr': 0.001, 'batch': 32, 'lookback': 30, 'activation': 'sigmoid'}

=== Generation 4/4 ===
Best SMAPE: 36.4640
Best params: {'units1': 16, 'dropout': 0.3, 'lr': 0.001, 'batch': 32, 'lookback': 30, 'activation': 'sigmoid'}

Best params: {'units1': 16, 'dropout': 0.3, 'lr': 0.001, 'batch': 32, 'lookback': 30, 'activation': 'sigmoid'}
Best SMAPE: 36.510765075683594


In [12]:
import os
os.makedirs(RESULT_DIR, exist_ok=True)

results_gao_summary = []
results_gao_detail  = []

for h in HORIZONS:
    print(f'\n=== [GAO] Horizon = {h} ===')
    print('  Building arrays...')

    X_num_tr, X_cats_tr, y_tr, X_num_vl, X_cats_vl, y_vl, scalers = \
        build_global_arrays(h, BEST["lookback"])

    print(f'  Train: {X_num_tr.shape} | Val: {X_num_vl.shape}')
    print(f'  Params: {BEST}')


    model = build_model(
        BEST["lookback"],
        len(NUM_COLS),
        cat_vocab_sizes,
        horizon=h,
        lstm_units=BEST["units1"],
        dropout=BEST["dropout"],
        activation=BEST["activation"]
    )

    cb = callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=BEST["lr"]),
        loss='mse'
    )

    model.fit(
        [X_num_tr] + X_cats_tr, y_tr,
        validation_data=([X_num_vl] + X_cats_vl, y_vl),
        epochs=50,
        batch_size=BEST["batch"],
        callbacks=[cb],
        verbose=0
    )

    print('  Rolling eval on TEST...')

    scores = {k: [] for k in ['smape', 'mase', 'rmse', 'rmsle']}

    for store, product in series_keys:
        key = f'{store}_{product}'

        r = rolling_eval(
            model,
            scalers[key],
            store,
            product,
            h,
            BEST["lookback"]
        )

        results_gao_detail.append({
            'model': 'LSTM-EntityEmbedding-GAO',
            'dataset': 'retail_inventory_daily',
            'target': TARGET,
            'horizon': h,
            'store': store,
            'product': product,
            'lookback': BEST["lookback"],
            'units1': BEST["units1"],
            'dropout': BEST["dropout"],
            'lr': BEST["lr"],
            'batch': BEST["batch"],
            'activation': BEST["activation"],
            'smape': r['smape'],
            'mase': r['mase'],
            'rmse': r['rmse'],
            'rmsle': r['rmsle'],
        })

        for k in scores:
            scores[k].append(r[k])

        print(f"    {store} | {product} | "
              f"sMAPE={r['smape']:.2f}% "
              f"MASE={r['mase']:.4f} "
              f"RMSE={r['rmse']:.2f} "
              f"RMSLE={r['rmsle']:.4f}")

    row = {
        'model':         'LSTM-EntityEmbedding-GAO',
        'dataset':       'retail_inventory_daily',
        'target':        TARGET,
        'horizon':       h,
        'lookback':      BEST["lookback"],
        'mean_smape':    round(float(np.nanmean(scores['smape'])), 4),
        'median_smape':  round(float(np.nanmedian(scores['smape'])), 4),
        'mean_mase':     round(float(np.nanmean(scores['mase'])), 4),
        'median_mase':   round(float(np.nanmedian(scores['mase'])), 4),
        'mean_rmse':     round(float(np.nanmean(scores['rmse'])), 4),
        'median_rmse':   round(float(np.nanmedian(scores['rmse'])), 4),
        'mean_rmsle':    round(float(np.nanmean(scores['rmsle'])), 4),
        'median_rmsle':  round(float(np.nanmedian(scores['rmsle'])), 4),
    }

    results_gao_summary.append(row)

    print(f"  H={h} | "
          f"sMAPE={row['mean_smape']:.2f}% "
          f"MASE={row['mean_mase']:.4f} "
          f"RMSE={row['mean_rmse']:.2f} "
          f"RMSLE={row['mean_rmsle']:.4f}")



summary_path = f'{RESULT_DIR}/lstm_entity_emb_gao_summary.csv'
detail_path  = f'{RESULT_DIR}/lstm_entity_emb_gao_detail.csv'

import pandas as pd
pd.DataFrame(results_gao_summary).to_csv(summary_path, index=False)
pd.DataFrame(results_gao_detail).to_csv(detail_path, index=False)

print(f'\nSaved summary to {summary_path}')
print(f'Saved detail  to {detail_path}')

pd.DataFrame(results_gao_summary)


=== [GAO] Horizon = 7 ===
  Building arrays...
  Train: (7300, 30, 19) | Val: (9100, 30, 19)
  Params: {'units1': 16, 'dropout': 0.3, 'lr': 0.001, 'batch': 32, 'lookback': 30, 'activation': 'sigmoid'}
  Rolling eval on TEST...
    S001 | P0001 | sMAPE=34.66% MASE=0.8377 RMSE=38.09 RMSLE=0.4600
    S001 | P0002 | sMAPE=29.27% MASE=0.6437 RMSE=34.23 RMSLE=0.4299
    S001 | P0003 | sMAPE=34.99% MASE=0.7225 RMSE=48.54 RMSLE=0.5317
    S001 | P0004 | sMAPE=39.62% MASE=0.7831 RMSE=35.54 RMSLE=0.6917
    S001 | P0005 | sMAPE=35.52% MASE=0.6947 RMSE=41.26 RMSLE=0.4553
    S001 | P0006 | sMAPE=44.60% MASE=0.6890 RMSE=41.13 RMSLE=0.8938
    S001 | P0007 | sMAPE=41.14% MASE=0.7784 RMSE=51.13 RMSLE=0.6072
    S001 | P0008 | sMAPE=33.90% MASE=0.7488 RMSE=31.61 RMSLE=0.4532
    S001 | P0009 | sMAPE=31.89% MASE=0.8577 RMSE=44.84 RMSLE=0.4159
    S001 | P0010 | sMAPE=32.85% MASE=0.6902 RMSE=28.21 RMSLE=0.4346
    S001 | P0011 | sMAPE=37.46% MASE=0.7528 RMSE=31.27 RMSLE=0.5055
    S001 | P0012 | sMAPE

,model,dataset,target,horizon,lookback,mean_smape,median_smape,mean_mase,median_mase,mean_rmse,median_rmse,mean_rmsle,median_rmsle
0,LSTM-EntityEmbedding-GAO,retail_inventory_daily,Units Sold,7,30,37.3175,37.5584,0.7485,0.7530,38.9760,39.4177,0.5957,0.5785
1,LSTM-EntityEmbedding-GAO,retail_inventory_daily,Units Sold,14,30,38.6357,38.6583,0.7741,0.7688,40.0093,40.6772,0.6158,0.5923
2,LSTM-EntityEmbedding-GAO,retail_inventory_daily,Units Sold,28,30,38.7668,39.2852,0.7809,0.7869,40.2494,40.2209,0.6232,0.6031
